SPDX-FileCopyrightText: © 2026 Idiap Research Institute <contact@idiap.ch>
SPDX-FileContributor: David Alonso del Barrio <ddbarrio@idiap.ch>
SPDX-License-Identifier: GPL-3.0-only

# EMN Factsheet OCR Extarction Pipeline

The notebook below demonstrates the automated extraction pipeline using a small sample of 5 PDFs included in this repository. The emn_18_cleaned.csv file provided in this repo represents the cleaned output for these 5 samples only, result of applying the pipeline below and performing the subsequent manual cleaning.

The full dataset presented in our paper (covering 35 countries and 13 years)is in our Zenodo repository: https://doi.org/10.34777/fxzr-nc56

In [1]:
import os
import fitz 
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
import re
import pandas as pd

# Paths
input_folder = "./data/2018/sample/" 
output_folder = "./data/2018/sample/output_pdfs_alternative/"  
os.makedirs(output_folder, exist_ok=True)  

### Extraction Process:
We show the extardction process from PDFs to tabualr data 

In [2]:
def make_pdf_searchable(input_folder, output_folder):
    """
    Converts image-based PDFs to searchable PDFs using OCR.
    
    This function processes each PDF in the input directory, performs OCR 
    via Tesseract to extract text, and embeds this text as an invisible 
    layer within the original PDF pages.
    
    Args:
        input_folder (str): Directory containing the source PDFs.
        output_folder (str): Directory where the searchable PDFs will be saved.
    """
    os.makedirs(output_folder, exist_ok=True)
    
    for pdf_filename in os.listdir(input_folder):
        if not pdf_filename.endswith(".pdf"):
            continue
            
        pdf_path = os.path.join(input_folder, pdf_filename)
        output_pdf_path = os.path.join(output_folder, f"{os.path.splitext(pdf_filename)[0]}_searchable.pdf")
        
        # Convert PDF pages to images for OCR processing
        images = convert_from_path(pdf_path)
        
        with fitz.open() as doc:
            for img in images:
                # Ensure RGB compatibility for PyMuPDF
                if img.mode != "RGB":
                    img = img.convert("RGB")

                # Perform OCR; text is added as an overlay for searchability
                ocr_text = pytesseract.image_to_string(img)
                
                # Create a page matching the image dimensions
                page = doc.new_page(width=img.width, height=img.height)
                img_bytes = img.tobytes("jpeg", "RGB")
                page.insert_image(page.rect, stream=img_bytes)
                page.insert_text((10, 10), ocr_text, fontname="helv", fontsize=8)

            doc.save(output_pdf_path)
            print(f"Successfully generated: {output_pdf_path}")

In [3]:
def extract_text_mupdf(pdf_path):
    """
    Parses text content from a PDF file using PyMuPDF.
    
    Args:
        pdf_path (str): File path to the source PDF.
        
    Returns:
        str: Combined text from all pages with preserved line breaks.
    """
    with fitz.open(pdf_path) as doc:
        text = "\n".join([page.get_text("text") for page in doc])
    return text

In [4]:
def format_text_section(text):
    """
    Normalizes raw text extracted from EMN factsheets.
    
    This function performs multi-stage cleaning, including:
    1. Extracting the core content between defined markers.
    2. Removing metadata artifacts (e.g., EMN-specific headers).
    3. Resolving line-broken headers and normalizing whitespace.
    4. Restructuring section-based content into a clean, paragraph-heavy format.
    
    Args:
        text (str): The raw text extracted from the PDF.
        
    Returns:
        str: The structured, cleaned text, or None if extraction fails.
    """
    # Isolate content within the relevant document boundaries
    match = re.search(r"(European.*?)(?:STATISTICAL ANNEX)", text, re.DOTALL)

    if not match:
        print("Warning: Could not find target section markers.")
        return None

    cleaned_text = match.group(1)

    # 1. Strip unwanted EMN metadata headers
    cleaned_text = re.sub(r"\d+\sEMN\sCountry\sFactsheet:\s[A-Z]+\n?", '', cleaned_text)

    # 2. Reconstruct split section titles and ensure title-newline formatting
    cleaned_text = re.sub(r"(\b[A-Z]+(?:\s+[A-Z]+)*)(\s*-\s*\n)([A-Z]+(?:\s+[A-Z]+)*)", r"\1 \3", cleaned_text)
    cleaned_text = re.sub(r"(?<!\n)([A-Z][A-Z ]{2,})\b", r"\n\1", cleaned_text)

    # 3. Clean up residual OCR artifacts and redundant whitespace
    cleaned_text = re.sub(r"(?<=\n)([A-Z])\s*\n([A-Z])", r"\1\2", cleaned_text)
    cleaned_text = re.sub(r"\n\s+|\s+\n", "\n", cleaned_text)

    # 4. Structure text into Title-Paragraph pairs
    sections = re.split(r"(\n[A-Z ]{2,}\n)", cleaned_text)
    final_text = []

    for section in sections:
        section = section.strip()
        if not section:
            continue
        # Format uppercase titles vs paragraph content
        if section.isupper():
            final_text.append(f"{section}")
        else:
            final_text.append(re.sub(r"\s+", " ", section))

    return "\n\n".join(final_text)

In [5]:
def process_pdfs(folder_path):
    """
    Orchestrates the extraction and formatting pipeline for a directory of PDFs.
    
    This function iterates through all PDFs, invokes the text extraction and 
    formatting modules, and aggregates the results into a structured DataFrame.
    
    Args:
        folder_path (str): Path to the directory containing source PDFs.
        
    Returns:
        pd.DataFrame: A DataFrame containing 'Country', 'Year', and 'Text' columns.
    """
    data = []
    
    # Ensure folder exists
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Directory not found: {folder_path}")

    for filename in os.listdir(folder_path):
        if not filename.endswith('.pdf'):
            continue
            
        pdf_path = os.path.join(folder_path, filename)
        
        # 1. Extraction
        raw_text = extract_text_mupdf(pdf_path)
        if not raw_text:
            print(f"Warning: Empty content in {filename}")
            continue
            
        # 2. Formatting
        formatted_text = format_text_section(raw_text)
        if not formatted_text:
            print(f"Skipped {filename}: Parsing failed.")
            continue
            
        # 3. Metadata extraction (Country and Year from filename)
        parts = filename.split('-')
        if len(parts) >= 4:
            data.append({
                'Country': parts[2],
                'Year': parts[3][:4],
                'Text': formatted_text
            })
        else:
            print(f"Warning: Unexpected filename format in {filename}")
    
    return pd.DataFrame(data)

In [6]:
if not os.path.exists(output_folder) or not os.listdir(output_folder):
    print("OCR output folder is empty. Running OCR...")
    make_pdf_searchable(input_folder, output_folder)
else:
    print("Searchable PDFs already exist. Skipping OCR.")

df = process_pdfs(output_folder)
df
# df.to_csv("extracted_factsheets.csv", index=False)

OCR output folder is empty. Running OCR...
Successfully generated: ./data/2018/sample/output_pdfs_alternative/EMN-factsheet-Belgium-2018_en_searchable.pdf
Successfully generated: ./data/2018/sample/output_pdfs_alternative/EMN-factsheet-Bulgaria-2018_en_searchable.pdf
Successfully generated: ./data/2018/sample/output_pdfs_alternative/EMN-factsheet-Croatia-2018_en_searchable.pdf
Successfully generated: ./data/2018/sample/output_pdfs_alternative/EMN-factsheet-Cyprus-2018_en_searchable.pdf
Successfully generated: ./data/2018/sample/output_pdfs_alternative/EMN-factsheet-Austria-2018_en_searchable.pdf


,Country,Year,Text
0,Austria,2018,European Commission\n\nAUSTRIA\n\nEMN COUNTRY ...
1,Cyprus,2018,European Commission\n\nCYPRUS\n\nEMN COUNTRY F...
2,Belgium,2018,European Commission No relevant updates were r...
3,Croatia,2018,European Commission Croatia\n\nEMN COUNTRY FAC...
4,Bulgaria,2018,European Commission The national legislation t...


### Visualizing Extraction Quality:
Below, we compare a sample of the raw output from our pipeline to the cleaned version. This illustrates the impact of the manual curation phase. This is the pdf of reference for this example: ./data/2018/sample/EMN-factsheet-Austria-2018_en.pdf 

In [7]:
print(df.Text[0])

European Commission

AUSTRIA

EMN COUNTRY FACTSHEET
MAIN DEVELOPMENTS IN MIGRATION AND INTERNATIONAL PROTECTION,

INCLUDING LATEST STATISTICS

LEGAL MIGRATION AND
(2)

MOBILITY

PROMOTING LEGAL MIGRATION CHANNELS The Act Amending the Aliens Law 2018 adopted legal changes across several migration areas. Among other things, it implemented the Students and Researchers Directive ((EU) 2016/801) in 2018. As a result, two new titles for temporary residence were instituted, one for researcher mobility and one for volunteers, and the evidence requirement for accommodation standards was dropped for foreigners applying for a student or research residence.

ECONOMIC MIGRATION

New regulations that entered into force at the beginning of 2018 expanded the list of shortage occupations for the employment of skilled workers

FAMILY REUNIFICATION

from third countries, from 11 to 27. Furthermore, the No relevant changes were reported in 2018. points system to obtain a Red-White-Red Card? for other key 

### Note on Data Quality
The raw extraction above produces the initial structure. 
As described in the paper, this dataset was subjected to a three-pass manual review and cleaning process. 
The cleaned version is loaded below, and we show the same example as above, but in this case all cleaned. Reference to evaluate the quality: ./data/2018/sample/EMN-factsheet-Austria-2018_en.pdf

In [8]:
df_18=pd.read_csv('./data/2018/final/emn_18_cleaned.csv')
print(df_18.text[0])

KEY POINTS
The Act Amending the Aliens Law 2018 was adopted, which included changes to legal and economic migration, international protection and citizenship. One of the changes introduced by the above-mentioned law was that officials were now authorised to seize and analyse media storage in the possession of asylum seekers to determine their travel route. A special policy focus was the return of third-country nationals staying in Austria irregularly. The rate of forced returns increased by 47%.

LEGAL MIGRATION AND MOBILITY
PROMOTING LEGAL MIGRATION CHANNELS
The Act Amending the Aliens Law 2018 adopted legal changes across several migration areas. Among other things, it implemented the Students and Researchers Directive ((EU) 2016/801) in 2018. As a result, two new titles for temporary residence were instituted, one for researcher mobility and one for volunteers, and the evidence requirement for accommodation standards was dropped for foreigners applying for a student or research resi